In [1]:
# ==========================================
# 0. CẤU HÌNH ĐƯỜNG DẪN (repo-relative)
# ==========================================
import os, re
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

BASE = os.getcwd()          # notebook nằm trong repo
OUT_DIR = os.path.join(BASE, "output")
os.makedirs(OUT_DIR, exist_ok=True)

# --- Nguồn dữ liệu (giai đoạn phân tích: T1-T2/2026) ---
F_LEAD = os.path.join(BASE, "Sales_Marketing dataset - SALE T1-2-3_2026.csv")
F_HEN  = os.path.join(BASE, "ĐĂT HẸN .csv")
F_INV  = os.path.join(BASE, "Doanh thu T11.2025 đến 25.02.xlsx")
F_HOADON = os.path.join(BASE, "kiotviet.csv")   # chỉ để các cell cũ (bên dưới) còn chạy được   # superset của kiotviet.csv
                                                                   # (1.011 HĐ vs 710, lùi về T11/2025)

# Cửa sổ phân tích funnel = nơi CẢ 3 nguồn cùng phủ.
# Hóa đơn T11/2025 vẫn được giữ để nhận diện khách mua lặp (CLV), nhưng
# không tính vào funnel vì chưa có lớp lead tương ứng.
WINDOW_START = pd.Timestamp("2026-01-01")

# LƯU Ý DATA: T12/2025 thiếu toàn bộ hóa đơn (0 dòng, kẹp giữa T11=302 và T1=484).
# Không thể export lại -> mọi số liệu chuỗi thời gian phải bỏ qua tháng này.

def clean_phone(phone):
    """Chuẩn hóa SĐT -> khóa join dạng 0XXXXXXXXX"""
    if pd.isna(phone) or str(phone).strip() == '': return np.nan
    p = str(phone).split('.')[0]; p = re.sub(r'\D', '', p)
    if not p: return np.nan
    if p.startswith('84'): p = '0' + p[2:]
    elif not p.startswith('0'): p = '0' + p
    return p

def clean_money(val):
    if pd.isna(val): return 0
    try: return int(float(str(val).replace('.', '').replace(',', '').strip()))
    except: return 0

def parse_date_vn(x):
    """dd/mm/yyyy. File lead có 35 dòng gõ nhầm năm 2025 -> ép về 2026."""
    d = pd.to_datetime(str(x).strip(), dayfirst=True, errors='coerce')
    if pd.notnull(d) and d.year == 2025 and d.month <= 3:
        d = d.replace(year=2026)
    return d

def load_leads():
    """Lead T1-T3/2026. Header có 'TRẠNG THÁI' lặp 3 lần -> pandas tự đánh số;
    cột thật dùng cho phân tích là TÌNH TRẠNG."""
    df = pd.read_csv(F_LEAD)
    df = df.dropna(how='all')
    df['SDT'] = df['SỐ ĐT']                       # alias tương thích các cell cũ
    print(f"[load_leads] {len(df)} dòng lead")
    return df

def load_appointments(df_lead):
    """Hẹn = HỢP NHẤT 2 nguồn: cột NGÀY HẸN trong file lead (309 SĐT)
    + file ĐĂT HẸN riêng (207 SĐT). Chỉ dùng 1 nguồn sẽ mất tới 106 SĐT."""
    a = df_lead.loc[df_lead['NGÀY HẸN'].notna(), ['SỐ ĐT', 'NGÀY HẸN']].copy()
    b = pd.read_csv(F_HEN)[['SỐ ĐT', 'NGÀY HẸN']].copy()
    h = pd.concat([a, b], ignore_index=True)
    h['Phone_Clean'] = h['SỐ ĐT'].apply(clean_phone)
    h = h.dropna(subset=['Phone_Clean']).drop_duplicates('Phone_Clean', keep='first')
    print(f"[load_appointments] {len(h)} SĐT có hẹn (hợp nhất 2 nguồn)")
    return h[['Phone_Clean', 'NGÀY HẸN']]

def load_invoices():
    d = pd.read_excel(F_INV, sheet_name=0)
    d['Phone_Clean'] = d['Điện thoại'].apply(clean_phone)
    d['Ngày HĐ'] = pd.to_datetime(d['Thời gian'], errors='coerce', dayfirst=True)
    d['Doanh Thu (VNĐ)'] = d['Khách cần trả'].apply(clean_money)
    print(f"[load_invoices] {len(d)} dòng, {d['Mã hóa đơn'].nunique()} HĐ "
          f"({d['Ngày HĐ'].min():%m/%Y} - {d['Ngày HĐ'].max():%m/%Y})")
    return d


In [2]:
import pandas as pd
import numpy as np
import re

# ==========================================
# 1. HÀM LÀM SẠCH SỐ ĐIỆN THOẠI
# ==========================================
def clean_phone(phone):
    if pd.isna(phone) or str(phone).strip() == '':
        return np.nan
    p = str(phone).split('.')[0]
    p = re.sub(r'\D', '', p)
    if not p:
        return np.nan
    if p.startswith('84'):
        p = '0' + p[2:]
    elif not p.startswith('0'):
        p = '0' + p
    return p

# ==========================================
# 2. ĐỌC DỮ LIỆU THÔ
# ==========================================
print("Đang đọc dữ liệu...")
df_lead = load_leads()
df_hen = pd.read_csv(F_HEN)
df_hoadon = pd.read_csv(F_HOADON)

# Lọc các cột cần thiết cho Lead ngay từ đầu
cols_lead = ['SỐ ĐT', 'LOẠI TIN NHẮN', 'NHÓM SP', 'CHATPAGE', 'NGUỒN', 'QUAN TÂM', 'TÌNH TRẠNG']
cols_lead = [c for c in cols_lead if c in df_lead.columns]
df_lead = df_lead[cols_lead]

# ==========================================
# 3. CHUẨN HÓA CỘT KEY (SỐ ĐIỆN THOẠI) & TÁCH DATA
# ==========================================
df_lead['Phone_Clean'] = df_lead['SỐ ĐT'].apply(clean_phone)
df_hen['Phone_Clean'] = df_hen['SỐ ĐT'].apply(clean_phone)
df_hoadon['Phone_Clean'] = df_hoadon['Điện thoại'].apply(clean_phone)

# TÁCH RIÊNG NHỮNG LEAD KHÔNG CÓ SĐT ĐỂ THÊM VÀO SAU
df_lead_no_phone = df_lead[df_lead['Phone_Clean'].isna()].copy()
df_lead_no_phone['In_Lead'] = True
df_lead_no_phone['Phân nhóm MECE'] = 'Nhóm 0: Lead chưa có SĐT'
df_lead_no_phone['Số lượng Lead tính'] = 1 # Đếm là 1 Lead

# CHỈ GIỮ LẠI CÁC DÒNG CÓ SĐT ĐỂ MERGE
df_lead_has_phone = df_lead.dropna(subset=['Phone_Clean'])
df_hen = df_hen.dropna(subset=['Phone_Clean'])
df_hoadon = df_hoadon.dropna(subset=['Phone_Clean'])

# Lọc cột cho Hẹn và Hóa đơn
cols_hen = ['Phone_Clean', 'NGÀY HẸN']
cols_hen = [c for c in cols_hen if c in df_hen.columns]

cols_hoadon = ['Phone_Clean', 'Mã hóa đơn', 'Mã khách hàng', 'Điện thoại', 'Khách cần trả', 'Tên hàng', 'Thời gian']
cols_hoadon = [c for c in cols_hoadon if c in df_hoadon.columns]

df_hen = df_hen[cols_hen]
df_hoadon = df_hoadon[cols_hoadon]

# ==========================================
# 4. XỬ LÝ TRÙNG LẶP TRƯỚC KHI MERGE
# ==========================================
df_lead_unique = df_lead_has_phone.drop_duplicates(subset=['Phone_Clean'], keep='first').copy()
df_lead_unique['In_Lead'] = True  

df_hen_unique = df_hen.drop_duplicates(subset=['Phone_Clean'], keep='first').copy()
df_hen_unique['In_Hen'] = True    

# ==========================================
# 5. GỘP DỮ LIỆU CÓ SĐT (OUTER JOIN)
# ==========================================
df_merge_1 = pd.merge(df_lead_unique, df_hoadon, on='Phone_Clean', how='outer')
df_final_has_phone = pd.merge(df_merge_1, df_hen_unique, on='Phone_Clean', how='outer')

# ==========================================
# 6. PHÂN NHÓM MECE CHO DATA CÓ SĐT
# ==========================================
def phan_nhom_mece(row):
    L = row['In_Lead'] == True
    H = row['In_Hen'] == True
    HD = pd.notna(row['Mã hóa đơn'])
    
    if not L and not H and HD:
        return 'Nhóm 1: Khách tự nhiên vãng lai (Chỉ có HĐ)'
    elif L and not H and not HD:
        return 'Nhóm 2: Lead có SĐT nhưng chưa hẹn & chưa chốt'
    elif L and not H and HD:
        return 'Nhóm 3: Lead chốt thẳng không cần hẹn'
    elif L and H and not HD:
        return 'Nhóm 4: Lead đặt hẹn nhưng rớt (bom lịch/ko mua)'
    elif L and H and HD:
        return 'Nhóm 5: Lead hoàn hảo (Đủ 3 bước)'
    else:
        return 'Khác'

df_final_has_phone['Phân nhóm MECE'] = df_final_has_phone.apply(phan_nhom_mece, axis=1)

# Xử lý đếm Lead trùng lặp cho khách có SĐT
if 'Thời gian' in df_final_has_phone.columns:
    df_final_has_phone = df_final_has_phone.sort_values(by=['Phone_Clean', 'Thời gian'], na_position='last')
df_final_has_phone['Số lượng Lead tính'] = (~df_final_has_phone.duplicated(subset=['Phone_Clean'])).astype(int)

# Đồng bộ hiển thị SĐT
df_final_has_phone['SĐT Cuối'] = df_final_has_phone['Phone_Clean'].apply(lambda x: f"'{x}" if pd.notna(x) else np.nan)

# ==========================================
# 7. GỘP LẠI (CONCAT) VỚI NHỮNG LEAD KHÔNG CÓ SĐT
# ==========================================
# Cột SĐT Cuối của Nhóm 0 sẽ để trống
df_final = pd.concat([df_final_has_phone, df_lead_no_phone], ignore_index=True)

# ==========================================
# 8. DỌN DẸP VÀ XUẤT FILE
# ==========================================
columns_to_drop = ['In_Lead', 'In_Hen', 'Điện thoại', 'Phone_Clean', 'SỐ ĐT']
df_final = df_final.drop(columns=[c for c in columns_to_drop if c in df_final.columns])

# Đưa cột quan trọng lên đầu
cols = df_final.columns.tolist()
cols_first = ['SĐT Cuối', 'Phân nhóm MECE', 'Số lượng Lead tính']
cols = cols_first + [c for c in cols if c not in cols_first]
df_final = df_final[cols]

# Xuất ra Excel
file_output = os.path.join(OUT_DIR, 'BaoCao_Gop_PhanTich_MECE.xlsx')
df_final.to_excel(file_output, index=False)
print(f"Xử lý thành công! File đã lưu tại: {file_output}")

Đang đọc dữ liệu...
[load_leads] 2439 dòng lead


Xử lý thành công! File đã lưu tại: /Users/tranvomanhtuan/Phun-Xam-Vic---Data-Analysis/output/BaoCao_Gop_PhanTich_MECE.xlsx


In [3]:
# ==========================================
# 9. BÁO CÁO KIỂM TOÁN DỮ LIỆU (DATA AUDIT REPORT)
# ==========================================
print("\n" + "="*50)
print("BÁO CÁO NGHIỆM THU MERGE DỮ LIỆU")
print("="*50)

# 1. ĐỐI SOÁT VĨ MÔ (Đảm bảo không bị rớt dữ liệu)
print("\n[1] KIỂM TRA TỔNG QUAN (MACRO)")

# Kiểm tra Hóa đơn (Quan trọng nhất: Không được rớt doanh thu)
hd_ban_dau = df_hoadon['Mã hóa đơn'].nunique()
hd_sau_merge = df_final_has_phone['Mã hóa đơn'].dropna().nunique()
print(f"- Tổng số Mã hóa đơn ở file gốc: {hd_ban_dau}")
print(f"- Tổng số Mã hóa đơn sau merge:  {hd_sau_merge}")
if hd_ban_dau == hd_sau_merge:
    print("  => CHUẨN XÁC: Không rớt bất kỳ hóa đơn nào!")
else:
    print(f"  => CẢNH BÁO: Lệch {abs(hd_ban_dau - hd_sau_merge)} hóa đơn!")

# Kiểm tra Lead có SĐT
lead_ban_dau = df_lead_unique['Phone_Clean'].nunique()
lead_sau_merge = df_final_has_phone[df_final_has_phone['Phân nhóm MECE'].str.contains('Lead')]['Phone_Clean'].nunique()
print(f"\n- Tổng số Lead (có SĐT) file gốc: {lead_ban_dau}")
print(f"- Tổng số Lead (có SĐT) sau merge: {lead_sau_merge}")
if lead_ban_dau == lead_sau_merge:
    print("  => CHUẨN XÁC: Không rớt SĐT Lead nào!")

# 2. KIỂM TRA LOGIC MECE
print("\n[2] PHÂN BỔ 6 NHÓM MECE")
print(df_final['Phân nhóm MECE'].value_counts().to_string())

# 3. TRUY VẾT CÁ NHÂN (MICRO)
print("\n[3] CÔNG CỤ TRUY VẾT SĐT NGẪU NHIÊN (MICRO)")

def truy_vet_sdt(sdt_can_tim):
    sdt_clean = clean_phone(sdt_can_tim)
    print(f"\nĐang truy vết SĐT: {sdt_can_tim} (Clean: '{sdt_clean}')")
    
    # Check sự tồn tại trong 3 file gốc
    in_lead = sdt_clean in df_lead_has_phone['Phone_Clean'].values
    in_hen = sdt_clean in df_hen['Phone_Clean'].values
    in_hd = sdt_clean in df_hoadon['Phone_Clean'].values
    
    print(f"- Tồn tại trong file Lead (Sale T1-2-3)? : {'CÓ' if in_lead else 'Không'}")
    print(f"- Tồn tại trong file Đặt Hẹn?            : {'CÓ' if in_hen else 'Không'}")
    print(f"- Tồn tại trong file Hóa Đơn?            : {'CÓ' if in_hd else 'Không'}")
    
    # Check kết quả sau merge
    ket_qua = df_final[df_final['Phone_Clean'] == sdt_clean]
    if not ket_qua.empty:
        nhom_mece = ket_qua['Phân nhóm MECE'].iloc[0]
        so_hd = ket_qua['Mã hóa đơn'].dropna().nunique()
        print(f"=> KẾT LUẬN MERGE: Khách được xếp đúng vào: [{nhom_mece}] với {so_hd} hóa đơn.")
    else:
        print("=> KẾT LUẬN: Không tìm thấy trong file kết quả!")

# Máy tính tự động bốc ngẫu nhiên 3 SĐT ở 3 nhóm khác nhau để test chéo
try:
    sdt_test_nhom1 = df_final[df_final['Phân nhóm MECE'].str.contains('Nhóm 1')]['Phone_Clean'].dropna().sample(1).iloc[0]
    sdt_test_nhom4 = df_final[df_final['Phân nhóm MECE'].str.contains('Nhóm 4')]['Phone_Clean'].dropna().sample(1).iloc[0]
    sdt_test_nhom5 = df_final[df_final['Phân nhóm MECE'].str.contains('Nhóm 5')]['Phone_Clean'].dropna().sample(1).iloc[0]

    truy_vet_sdt(sdt_test_nhom1) # Khách vãng lai
    truy_vet_sdt(sdt_test_nhom4) # Khách đặt hẹn rớt
    truy_vet_sdt(sdt_test_nhom5) # Khách hoàn hảo
except Exception as e:
    print("\nKhông đủ data để test ngẫu nhiên, hoặc có nhóm không có SĐT nào.")


BÁO CÁO NGHIỆM THU MERGE DỮ LIỆU

[1] KIỂM TRA TỔNG QUAN (MACRO)
- Tổng số Mã hóa đơn ở file gốc: 702
- Tổng số Mã hóa đơn sau merge:  702
  => CHUẨN XÁC: Không rớt bất kỳ hóa đơn nào!

- Tổng số Lead (có SĐT) file gốc: 1300
- Tổng số Lead (có SĐT) sau merge: 1300
  => CHUẨN XÁC: Không rớt SĐT Lead nào!

[2] PHÂN BỔ 6 NHÓM MECE
Phân nhóm MECE
Nhóm 0: Lead chưa có SĐT                            1115
Nhóm 2: Lead có SĐT nhưng chưa hẹn & chưa chốt      1016
Nhóm 1: Khách tự nhiên vãng lai (Chỉ có HĐ)          443
Nhóm 5: Lead hoàn hảo (Đủ 3 bước)                    157
Nhóm 3: Lead chốt thẳng không cần hẹn                101
Nhóm 4: Lead đặt hẹn nhưng rớt (bom lịch/ko mua)      94
Khác                                                   3

[3] CÔNG CỤ TRUY VẾT SĐT NGẪU NHIÊN (MICRO)

Không đủ data để test ngẫu nhiên, hoặc có nhóm không có SĐT nào.


In [4]:
import pandas as pd
import numpy as np

# Đọc file kết quả (Hãy đảm bảo tên file đúng với file trên máy bạn)
df = pd.read_excel(os.path.join(OUT_DIR, 'BaoCao_Gop_PhanTich_MECE.xlsx'))

# ==========================================
# 1. CHUẨN HÓA DOANH THU (KHÁCH CẦN TRẢ)
# ==========================================
def clean_money(val):
    if pd.isna(val): return 0
    return int(str(val).replace('.', '').strip())
df['Doanh Thu (VNĐ)'] = df['Khách cần trả'].apply(clean_money)

# ==========================================
# 2. BÙ ĐẮP DỮ LIỆU NHÓM 1
# ==========================================
# Nhóm 1 (Khách vãng lai) sẽ bị thiếu thông tin ở một số cột
idx_nhom1 = df['Phân nhóm MECE'].str.contains('Nhóm 1', na=False)

# Cột loại tin nhắn điền 'Khác'
df.loc[idx_nhom1 & df['LOẠI TIN NHẮN'].isna(), 'LOẠI TIN NHẮN'] = 'Khác'
# Cột nguồn điền 'Khác'
df.loc[idx_nhom1 & df['NGUỒN'].isna(), 'NGUỒN'] = 'Khác'

# Cột NHÓM SP: Suy luận từ Tên Hàng nếu để trống
def fill_nhom_sp(row):
    if pd.notna(row['NHÓM SP']):
        return str(row['NHÓM SP']).strip().upper()
    ten_hang = str(row['Tên hàng']).upper()
    if 'KHÓA HỌC' in ten_hang or 'ĐÀO TẠO' in ten_hang or 'PLASMA ONLINE' in ten_hang:
        return 'ĐÀO TẠO'
    return 'DỊCH VỤ'
df['NHÓM SP_Clean'] = df.apply(fill_nhom_sp, axis=1)

# ==========================================
# 3. GOM NHÓM NGUỒN KHÁCH HÀNG
# ==========================================
def phan_loai_nguon(row):
    chatpage = str(row['CHATPAGE']).upper() if pd.notna(row['CHATPAGE']) else ""
    nguon = str(row['NGUỒN']).upper() if pd.notna(row['NGUỒN']) else ""
    mece = str(row['Phân nhóm MECE']).upper()
    
    combined = chatpage + " | " + nguon
    
    # Facebook
    if any(x in combined for x in ['FB CÔ HƯỜNG', 'FANPAGE PXV', 'FANPAGE HỌC VIỆN', 'FANPAGE', 'FACEBOOK', 'FB']):
        return 'Facebook'
    # Tiktok
    if any(x in combined for x in ['TIKTOK PXV', 'TIKTOK HỌC VIỆN', 'TIKTOK']):
        return 'Tiktok'
    # Hotline/Zalo
    if any(x in combined for x in ['HOTLINE', 'ZALO']):
        return 'Hotline/Zalo'
    # Khách cũ
    if any(x in combined for x in ['KHÁCH CŨ', 'DATA CŨ 2023', 'FILE CŨ']):
        return 'Khách cũ'
    # Vãng lai (nếu tự ghi là vãng lai, hoặc thuộc Nhóm 1 của MECE)
    if any(x in combined for x in ['VÃNG LAI']) or ('NHÓM 1' in mece):
        return 'Vãng lai'
    # Khác
    if any(x in combined for x in ['PAGE LX', 'INSTAGRAM', 'INSTGRAM']):
        return 'Khác'
        
    return 'Khác'
df['Kênh Tiếp Cận'] = df.apply(phan_loai_nguon, axis=1)

# ==========================================
# 4. CHUẨN HÓA SẢN PHẨM & LỌC DỊCH VỤ PHỄU
# ==========================================
def lay_sp_cot_loi(row):
    sp = str(row['Tên hàng']) if pd.notna(row['Tên hàng']) else str(row['QUAN TÂM'])
    sp = sp.upper()
    
    if 'XOÁ' in sp or 'XÓA' in sp: return 'Xóa Laser'
    if 'MÀY' in sp or 'TẠO SỢI' in sp or 'ĐIÊU KHẮC' in sp: return 'Làm Mày'
    if 'MÔI' in sp: return 'Làm Môi'
    if 'CO2' in sp or 'BÓC TÁCH' in sp: return 'CO2 / Cắt đáy sẹo'
    if 'PLASMA' in sp and 'KHÓA' in sp: return 'Khóa học'
    if 'MÍ' in sp or 'EYELINER' in sp: return 'Làm Mí'
    if pd.isna(row['Tên hàng']) and pd.isna(row['QUAN TÂM']): return 'Chưa rõ'
    return 'Dịch vụ/Sản phẩm khác'

# Cột Phân loại sản phẩm (giữ nguyên cột Tên hàng gốc)
df['Phân loại sản phẩm'] = df.apply(lay_sp_cot_loi, axis=1)

# Lọc Dịch vụ phễu (Trả về True/False)
funnel_services = [
    '(DV) GÓI TIẾT KIỆM - XÓA CHÂN MÀY TRỌN GÓI',
    '(DV) Xóa lần 01 - Chân Mày',
    '(DV) Công nghệ CO2 Fractional - Tách thâm',
    '(DV) Công nghệ CO2 Fractional - Gói Double bóc tách',
    '(DV) Công nghệ CO2 Fractional - Gói Double bóc tách (trọn gói)'
]
df['[Dịch vụ phễu]'] = df['Tên hàng'].isin(funnel_services)

# ==========================================
# 5. TẠO CHỈ SỐ PHỄU (FUNNEL METRICS) 1/0
# ==========================================
def tinh_pheu(row):
    if row['Số lượng Lead tính'] == 0:
        return pd.Series([0, 0, 0, 0]) # Bỏ qua các dòng hóa đơn lặp lại của cùng 1 người
    
    b1_mess = 1
    b2_sdt = 0 if 'Nhóm 0' in str(row['Phân nhóm MECE']) else 1
    b3_hen = 1 if ('Nhóm 4' in str(row['Phân nhóm MECE']) or 'Nhóm 5' in str(row['Phân nhóm MECE']) or pd.notna(row['NGÀY HẸN'])) else 0
    b4_don = 1 if ('Nhóm 3' in str(row['Phân nhóm MECE']) or 'Nhóm 5' in str(row['Phân nhóm MECE']) or 'Nhóm 1' in str(row['Phân nhóm MECE']) or pd.notna(row['Mã hóa đơn'])) else 0
    
    return pd.Series([b1_mess, b2_sdt, b3_hen, b4_don])

df[['[F] 1_Có Inbox', '[F] 2_Có SĐT', '[F] 3_Có Đặt Lịch', '[F] 4_Có Ra Đơn']] = df.apply(tinh_pheu, axis=1)

# ==========================================
# LƯU FILE KẾT QUẢ
# ==========================================
file_output = os.path.join(OUT_DIR, 'Data_Dashboard_Ready_v2.xlsx') 
# Nhớ đổi đường dẫn tuyệt đối như hồi nãy nếu chạy trên Macbook nhé:

df.to_excel(file_output, index=False)
print(f"Xử lý thành công! Đã lưu file: {file_output}")

Xử lý thành công! Đã lưu file: /Users/tranvomanhtuan/Phun-Xam-Vic---Data-Analysis/output/Data_Dashboard_Ready_v2.xlsx


In [5]:
import pandas as pd
import numpy as np
import re

# ==========================================
# 1. HÀM LÀM SẠCH SỐ ĐIỆN THOẠI
# ==========================================
def clean_phone(phone):
    if pd.isna(phone) or str(phone).strip() == '':
        return np.nan
    p = str(phone).split('.')[0]
    p = re.sub(r'\D', '', p)
    if not p:
        return np.nan
    if p.startswith('84'):
        p = '0' + p[2:]
    elif not p.startswith('0'):
        p = '0' + p
    return p

# ==========================================
# 2. ĐỌC DỮ LIỆU
# ==========================================
print("Đang đọc dữ liệu...")
df_dashboard = pd.read_excel(os.path.join(OUT_DIR, 'Data_Dashboard_Ready_v2.xlsx')) 
df_lead = load_leads()

# ==========================================
# 3. TẠO KEY KHỚP SĐT CHO 2 BẢNG
# ==========================================
df_dashboard['Phone_Key'] = df_dashboard['SĐT Cuối'].apply(clean_phone)
df_lead['Phone_Key'] = df_lead['SỐ ĐT'].apply(clean_phone)

# ==========================================
# 4. TRÍCH XUẤT VÀ ÉP BUỘC ĐỊNH DẠNG NGÀY LEAD (CHỈ 2026)
# ==========================================
df_lead_date = df_lead.dropna(subset=['Phone_Key', 'NGÀY']).copy()

def parse_lead_date_strict(date_str):
    try:
        d_str = str(date_str).strip()
        # Bước 1: Parse lấy ngày và tháng (ưu tiên dd/mm/yyyy hoặc dd/mm)
        d = pd.to_datetime(d_str, format='%d/%m/%Y', errors='coerce')
        if pd.isnull(d):
            d = pd.to_datetime(d_str, format='%d/%m', errors='coerce')
        if pd.isnull(d):
            # Fallback cho các định dạng ngày khác
            d = pd.to_datetime(d_str, dayfirst=True, errors='coerce')
            
        # Bước 2: ÉP BUỘC năm là 2026 (Diệt trừ lỗi gõ nhầm 2025)
        if pd.notnull(d):
            return d.replace(year=2026)
        return pd.NaT
    except:
        return pd.NaT

df_lead_date['Ngay_Lead_Original'] = df_lead_date['NGÀY'].apply(parse_lead_date_strict)

# Bỏ dòng lỗi ngày, sắp xếp và chỉ lấy ngày Lead TỚI SỚM NHẤT của 1 SĐT
df_lead_date = df_lead_date.dropna(subset=['Ngay_Lead_Original'])
df_lead_date = df_lead_date.sort_values('Ngay_Lead_Original').drop_duplicates('Phone_Key', keep='first')
df_lead_date = df_lead_date[['Phone_Key', 'Ngay_Lead_Original']]

# ==========================================
# 5. MERGE VÀO DASHBOARD VÀ TÍNH THỜI GIAN
# ==========================================
df_final = pd.merge(df_dashboard, df_lead_date, on='Phone_Key', how='left')

# Chuyển cột Thời gian (Ngày ra đơn) về datetime
# Lưu ý: Nếu hóa đơn cũng bị gõ nhầm 2025, ta cũng ép nó về 2026 để trừ không bị âm/sai
def fix_hoadon_year(hd_time):
    try:
        d = pd.to_datetime(hd_time, format='%d/%m/%Y %H:%M:%S', errors='coerce')
        if pd.notnull(d) and d.year == 2025:
            return d.replace(year=2026)
        return d
    except:
        return pd.NaT

df_final['Ngay_Hoa_Don'] = df_final['Thời gian'].apply(fix_hoadon_year)

# Tính Thời gian ra đơn (Ngày)
df_final['Thời gian ra đơn (Ngày)'] = (df_final['Ngay_Hoa_Don'] - df_final['Ngay_Lead_Original']).dt.total_seconds() / (24 * 3600)

# Làm tròn 1 chữ số thập phân
df_final['Thời gian ra đơn (Ngày)'] = df_final['Thời gian ra đơn (Ngày)'].round(1)

# Fix số âm thành 0
df_final.loc[df_final['Thời gian ra đơn (Ngày)'] < 0, 'Thời gian ra đơn (Ngày)'] = 0

# ==========================================
# 6. DỌN DẸP VÀ XUẤT FILE
# ==========================================
match_count = df_final['Thời gian ra đơn (Ngày)'].notna().sum()

columns_to_drop = ['Phone_Key', 'Ngay_Hoa_Don']
df_final = df_final.drop(columns=[c for c in columns_to_drop if c in df_final.columns])

file_out = os.path.join(OUT_DIR, 'Data_Dashboard_Final_With_Time.xlsx')
df_final.to_excel(file_out, index=False)

print(f"\nKiểm tra: Đã match thành công {match_count} đơn hàng có lịch sử Lead!")
print(f"File đã sẵn sàng tại: {file_out}")

Đang đọc dữ liệu...


[load_leads] 2439 dòng lead



Kiểm tra: Đã match thành công 0 đơn hàng có lịch sử Lead!
File đã sẵn sàng tại: /Users/tranvomanhtuan/Phun-Xam-Vic---Data-Analysis/output/Data_Dashboard_Final_With_Time.xlsx


In [6]:
# ==========================================
# PIPELINE CHÍNH — gộp 3 nguồn, MECE, funnel, CLV (cửa sổ T1-T2/2026)
# ==========================================
df_lead = load_leads()
df_hen  = load_appointments(df_lead)
df_inv  = load_invoices()

# --- 1. Chuẩn hóa lead ---
df_lead['Phone_Clean'] = df_lead['SỐ ĐT'].apply(clean_phone)
df_lead['Ngày Lead']   = df_lead['NGÀY'].apply(parse_date_vn)
df_lead = df_lead[df_lead['Ngày Lead'] >= WINDOW_START]

KEEP = ['Phone_Clean','Ngày Lead','TÊN KHÁCH HÀNG','LOẠI TIN NHẮN','NHÓM SP',
        'CHATPAGE','NGUỒN','QUAN TÂM','TÌNH TRẠNG','BÀI QC']
df_lead = df_lead[[c for c in KEEP if c in df_lead.columns]]

# Lead KHÔNG có SĐT: mỗi inbox = 1 lead (theo cách đếm đã chốt)
lead_no_phone = df_lead[df_lead['Phone_Clean'].isna()].copy()
# Lead CÓ SĐT: gộp 1 dòng/khách, lấy lần inbox sớm nhất
lead_phone = (df_lead.dropna(subset=['Phone_Clean'])
                     .sort_values('Ngày Lead')
                     .drop_duplicates('Phone_Clean', keep='first'))
lead_phone['In_Lead'] = True
df_hen = df_hen.copy(); df_hen['In_Hen'] = True

# --- 2. Hóa đơn: tách cửa sổ funnel vs lịch sử mua trước ---
inv_cols  = ['Phone_Clean','Mã hóa đơn','Mã khách hàng','Tên hàng','Ngày HĐ','Doanh Thu (VNĐ)']
inv_win   = df_inv[df_inv['Ngày HĐ'] >= WINDOW_START][inv_cols].dropna(subset=['Phone_Clean'])
inv_prior = df_inv[df_inv['Ngày HĐ'] <  WINDOW_START][inv_cols].dropna(subset=['Phone_Clean'])
prior_buyers = set(inv_prior['Phone_Clean'])

# --- 3. Outer join 3 nguồn qua SĐT ---
m = pd.merge(lead_phone, inv_win, on='Phone_Clean', how='outer')
m = pd.merge(m, df_hen, on='Phone_Clean', how='outer')

# --- 4. Phân nhóm MECE ---
def mece(r):
    """Vét cạn 8 tổ hợp của (Lead, Hẹn, Hóa đơn) — không để tổ hợp nào rơi vào
    nhánh sai. Trước đây 'hẹn mồ côi' đặt cuối nên nuốt nhầm case có hóa đơn."""
    L = r['In_Lead'] == True; H = r['In_Hen'] == True; HD = pd.notna(r['Mã hóa đơn'])
    if L:
        if H:  return 'Nhóm 5: Lead hoàn hảo (đủ 3 bước)' if HD else 'Nhóm 4: Đặt hẹn nhưng rớt'
        return 'Nhóm 3: Chốt thẳng không cần hẹn' if HD else 'Nhóm 2: Lead chưa hẹn & chưa chốt'
    # Không có hồ sơ lead:
    if H:  return 'Nhóm 6: Hẹn thiếu hồ sơ lead (lỗi data)'
    return 'Nhóm 1: Vãng lai (chỉ có HĐ)' if HD else 'Khác'
m['Phân nhóm MECE'] = m.apply(mece, axis=1)

# --- 5. Làm giàu thuộc tính ---
def kenh(r):
    c = (str(r.get('CHATPAGE')) + ' | ' + str(r.get('NGUỒN'))).upper()
    if 'TIKTOK' in c: return 'Tiktok'
    if any(x in c for x in ['FANPAGE','FACEBOOK','FB CÔ HƯỜNG']): return 'Facebook'
    if 'INSTGRAM' in c or 'INSTAGRAM' in c: return 'Instagram'
    if any(x in c for x in ['HOTLINE','ZALO']): return 'Hotline/Zalo'
    if any(x in c for x in ['KHÁCH CŨ','DATA CŨ','FILE CŨ']): return 'Khách cũ'
    if 'GIỚI THIỆU' in c: return 'Giới thiệu'
    if 'Nhóm 1' in str(r.get('Phân nhóm MECE','')): return 'Vãng lai (không rõ nguồn)'
    return 'Khác'
m['Kênh Tiếp Cận'] = m.apply(kenh, axis=1)

def nhom_sp(r):
    if pd.notna(r.get('NHÓM SP')): return str(r['NHÓM SP']).strip().upper()
    t = str(r.get('Tên hàng')).upper()
    return 'ĐÀO TẠO' if any(x in t for x in ['KHÓA HỌC','ĐÀO TẠO','PLASMA ONLINE']) else 'DỊCH VỤ'
m['NHÓM SP_Clean'] = m.apply(nhom_sp, axis=1)

def loai_sp(r):
    s = (str(r['Tên hàng']) if pd.notna(r.get('Tên hàng')) else str(r.get('QUAN TÂM'))).upper()
    if 'XOÁ' in s or 'XÓA' in s: return 'Xóa Laser'
    if any(x in s for x in ['MÀY','TẠO SỢI','ĐIÊU KHẮC']): return 'Làm Mày'
    if 'MÔI' in s: return 'Làm Môi'
    if 'CO2' in s or 'BÓC TÁCH' in s: return 'CO2 / Cắt đáy sẹo'
    if 'PLASMA' in s and 'KHÓA' in s: return 'Khóa học'
    if 'MÍ' in s or 'EYELINER' in s: return 'Làm Mí'
    return 'Chưa rõ' if s in ('NAN','') else 'Dịch vụ/SP khác'
m['Phân loại sản phẩm'] = m.apply(loai_sp, axis=1)

# Dịch vụ mồi/phễu — danh sách giữ nguyên theo xác nhận nghiệp vụ
FUNNEL_RE = r'GÓI TIẾT KIỆM|Xóa lần 01|CO2 Fractional'
m['[Dịch vụ mồi]'] = m['Tên hàng'].astype(str).str.contains(FUNNEL_RE, case=False, na=False)

# --- 6. Cờ funnel — mỗi SĐT chỉ đếm 1 lần ---
m = m.sort_values(['Phone_Clean','Ngày HĐ'], na_position='first')
first = ~m.duplicated('Phone_Clean', keep='first')
# Phễu chỉ tính trên LEAD (khách từng inbox). Vãng lai (Nhóm 1) không thuộc
# phễu marketing vì chưa từng nhắn tin -> tách riêng, không làm phình F4.
is_lead = m['In_Lead'] == True
m['[F] 1_Có Inbox']    = np.where(is_lead, 1, 0)
m['[F] 2_Có SĐT']      = np.where(is_lead & m['Phone_Clean'].notna(), 1, 0)
m['[F] 3_Có Đặt Lịch'] = np.where(is_lead & (m['In_Hen'] == True), 1, 0)
m['[F] 4_Có Ra Đơn']   = np.where(is_lead & m['Mã hóa đơn'].notna(), 1, 0)
m['[Vãng lai] Có Ra Đơn'] = np.where(~is_lead & m['Mã hóa đơn'].notna(), 1, 0)
m['Số lượng Lead tính'] = 1
for c in ['[F] 1_Có Inbox','[F] 2_Có SĐT','[F] 3_Có Đặt Lịch','[F] 4_Có Ra Đơn',
          '[Vãng lai] Có Ra Đơn','Số lượng Lead tính']:
    m.loc[~first, c] = 0

# --- 7. Chỉ số phái sinh ---
m['NGÀY HẸN'] = m['NGÀY HẸN'].apply(parse_date_vn)
m['Thời gian ra đơn (Ngày)'] = ((m['Ngày HĐ'] - m['Ngày Lead']).dt.total_seconds()/86400).round(1)
m.loc[m['Thời gian ra đơn (Ngày)'] < 0, 'Thời gian ra đơn (Ngày)'] = 0
m['Khách mua trước T1/2026'] = m['Phone_Clean'].isin(prior_buyers)
m['SĐT Cuối'] = m['Phone_Clean'].apply(lambda x: f"'{x}" if pd.notna(x) else np.nan)

# --- 8. Nối lead không SĐT (Nhóm 0) ---
lead_no_phone = lead_no_phone.assign(**{
    'Phân nhóm MECE':'Nhóm 0: Lead chưa có SĐT', '[F] 1_Có Inbox':1, '[F] 2_Có SĐT':0,
    '[F] 3_Có Đặt Lịch':0, '[F] 4_Có Ra Đơn':0, 'Số lượng Lead tính':1,
    'Doanh Thu (VNĐ)':0, '[Dịch vụ mồi]':False, 'Khách mua trước T1/2026':False,
    '[Vãng lai] Có Ra Đơn':0})
lead_no_phone['Kênh Tiếp Cận']      = lead_no_phone.apply(kenh, axis=1)
lead_no_phone['NHÓM SP_Clean']      = lead_no_phone.apply(
    lambda r: str(r['NHÓM SP']).strip().upper() if pd.notna(r.get('NHÓM SP')) else 'DỊCH VỤ', axis=1)
lead_no_phone['Phân loại sản phẩm'] = lead_no_phone.apply(loai_sp, axis=1)

df_master = pd.concat([m, lead_no_phone], ignore_index=True)
df_master = df_master.drop(columns=[c for c in ['In_Lead','In_Hen','Phone_Clean'] if c in df_master])

FIRST = ['SĐT Cuối','Ngày Lead','Ngày HĐ','NGÀY HẸN','Phân nhóm MECE','Kênh Tiếp Cận',
         'Doanh Thu (VNĐ)','Số lượng Lead tính','Thời gian ra đơn (Ngày)']
df_master = df_master[[c for c in FIRST if c in df_master] +
                      [c for c in df_master.columns if c not in FIRST]]

out_path = os.path.join(OUT_DIR, 'Master_Pipeline_2026.xlsx')
with pd.ExcelWriter(out_path, datetime_format='DD/MM/YYYY') as w:
    df_master.to_excel(w, index=False)

print(f"\n{'='*58}\n✅ MASTER 2026 — {len(df_master):,} dòng\n{'='*58}")
print(f"Cửa sổ funnel : T1-T2/2026  (T12/2025 thiếu hóa đơn — đã loại)")
print(f"Doanh thu     : {df_master['Doanh Thu (VNĐ)'].sum():,.0f} đ")
print(f"Hóa đơn       : {df_master['Mã hóa đơn'].nunique():,}")
print("\nPhễu lead (khử trùng theo SĐT, không gồm vãng lai):")
prev = None
for c in ['[F] 1_Có Inbox','[F] 2_Có SĐT','[F] 3_Có Đặt Lịch','[F] 4_Có Ra Đơn']:
    v = int(df_master[c].sum())
    step = f"  ({v/prev*100:5.1f}% so với bước trước)" if prev else ""
    print(f"  {c:22}: {v:6,}{step}")
    prev = v
f1 = int(df_master['[F] 1_Có Inbox'].sum()); f4 = int(df_master['[F] 4_Có Ra Đơn'].sum())
print(f"  => Tỷ lệ chốt trên tổng inbox: {f4/f1*100:.2f}%")
print(f"  Khách vãng lai ra đơn (ngoài phễu): {int(df_master['[Vãng lai] Có Ra Đơn'].sum()):,}")
print(f"\nMECE:\n{df_master['Phân nhóm MECE'].value_counts().to_string()}")
print(f"\nFile: {out_path}")


In [7]:
import pandas as pd

print("==========================================================")
print("🔍 BÁO CÁO KIỂM TRA CHẤT LƯỢNG DỮ LIỆU (DATA QUALITY CHECK)")
print("==========================================================\n")

# Đọc file kết quả sau khi chạy Pipeline (Hãy đảm bảo đường dẫn này đúng)
file_path = os.path.join(OUT_DIR, 'Data_Dashboard_FINAL_PIPELINE.xlsx')
try:
    df_check = pd.read_excel(file_path)
    print(f"Đã nạp file thành công. Tổng số dòng dữ liệu: {len(df_check)}\n")
except Exception as e:
    print(f"❌ Lỗi đọc file: {e}")
    exit()

# ---------------------------------------------------------
# TEST 1: KIỂM TRA LỖI LỌT DOANH THU 2025
# ---------------------------------------------------------
df_check['Thời_gian_Chuẩn_dt'] = pd.to_datetime(df_check['Thời_gian_Chuẩn'], errors='coerce')
years_found = df_check['Thời_gian_Chuẩn_dt'].dt.year.dropna().unique()

print("TEST 1: Kiểm tra ranh giới thời gian (Time Boundary)")
print(f"- Các năm có phát sinh doanh thu trong file: {list(years_found)}")
if 2025 in years_found:
    print("  ❌ LỖI NGHIÊM TRỌNG: Vẫn còn lọt hóa đơn năm 2025 vào báo cáo!")
else:
    print("  ✅ PASS: Doanh thu ảo của năm 2025 đã bị dọn sạch.")

# ---------------------------------------------------------
# TEST 2: KIỂM TRA BẢO TOÀN KHÁCH RỚT PHỄU
# ---------------------------------------------------------
lead_rot_pheu = df_check['Thời_gian_Chuẩn'].isna().sum()
print("\nTEST 2: Kiểm tra dữ liệu Lead chưa chốt đơn")
print(f"- Số dòng Lead không có ngày hóa đơn: {lead_rot_pheu} dòng")
if lead_rot_pheu == 0:
    print("  ❌ LỖI NGHIÊM TRỌNG: Tất cả khách rớt phễu đã bị xóa nhầm!")
else:
    print("  ✅ PASS: Dữ liệu khách hàng chưa chốt được bảo toàn hoàn hảo.")

# ---------------------------------------------------------
# TEST 3: KIỂM TRA NHÂN BẢN PHỄU (DEDUPLICATION)
# ---------------------------------------------------------
# Bỏ qua nhóm không có SĐT (Vãng lai) vì nhóm này vốn dĩ đã là các dòng độc lập
df_co_sdt = df_check[df_check['SĐT Cuối'].notna()]
max_inbox = df_co_sdt.groupby('SĐT Cuối')['[F] 1_Có Inbox'].sum().max()
max_don = df_co_sdt.groupby('SĐT Cuối')['[F] 4_Có Ra Đơn'].sum().max()

print("\nTEST 3: Kiểm tra đếm trùng lặp Phễu")
print(f"- Số lượt tính 'Inbox' tối đa cho 1 SĐT: {max_inbox}")
print(f"- Số lượt tính 'Ra đơn' tối đa cho 1 SĐT: {max_don}")

if max_inbox > 1 or max_don > 1:
    print("  ❌ LỖI NGHIÊM TRỌNG: Có khách hàng bị đếm nhiều lần vào Phễu!")
else:
    print("  ✅ PASS: Mỗi SĐT chỉ được đếm 1 lần duy nhất, chống nhân bản thành công.")

# ---------------------------------------------------------
# TEST 4: TỔNG KẾT CON SỐ THẬT
# ---------------------------------------------------------
total_rev = df_check['Doanh Thu (VNĐ)'].sum()
total_inbox = df_check['[F] 1_Có Inbox'].sum()

print("\n📊 TỔNG KẾT SỐ LIỆU ĐƯA LÊN LOOKER STUDIO:")
print(f"- Tổng Doanh Thu 2026: {total_rev:,.0f} VNĐ")
print(f"- Tổng Khách Inbox: {total_inbox:,.0f} khách")
print("==========================================================")

🔍 BÁO CÁO KIỂM TRA CHẤT LƯỢNG DỮ LIỆU (DATA QUALITY CHECK)



Đã nạp file thành công. Tổng số dòng dữ liệu: 6923

TEST 1: Kiểm tra ranh giới thời gian (Time Boundary)
- Các năm có phát sinh doanh thu trong file: [2026.0]
  ✅ PASS: Doanh thu ảo của năm 2025 đã bị dọn sạch.

TEST 2: Kiểm tra dữ liệu Lead chưa chốt đơn
- Số dòng Lead không có ngày hóa đơn: 6220 dòng
  ✅ PASS: Dữ liệu khách hàng chưa chốt được bảo toàn hoàn hảo.

TEST 3: Kiểm tra đếm trùng lặp Phễu
- Số lượt tính 'Inbox' tối đa cho 1 SĐT: 1
- Số lượt tính 'Ra đơn' tối đa cho 1 SĐT: 1
  ✅ PASS: Mỗi SĐT chỉ được đếm 1 lần duy nhất, chống nhân bản thành công.

📊 TỔNG KẾT SỐ LIỆU ĐƯA LÊN LOOKER STUDIO:
- Tổng Doanh Thu 2026: 2,504,224,000 VNĐ
- Tổng Khách Inbox: 6,724 khách
